In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
# ติดตั้ง rpy2 (เชื่อมระหว่าง Python และ R)
!pip install rpy2

# โหลด extension เพื่อใช้ R ใน Jupyter/Colab
%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R
# ติดตั้งแพ็กเกจ (รันแค่ครั้งเดียว)
install.packages("INLA", repos=c(getOption("repos"),
        INLA="https://inla.r-inla-download.org/R/stable"), dep=TRUE)
install.packages("spdep")
install.packages("sf")
install.packages("tidyverse")
install.packages("readxl")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘gsl’, ‘runjags’

trying URL 'https://cran.rstudio.com/src/contrib/gsl_2.1-8.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/runjags_2.2.2-5.tar.gz'
trying URL 'https://inla.r-inla-download.org/R/stable/src/contrib/INLA_24.12.11.tar.gz'

The downloaded source packages are in
	‘/tmp/Rtmpl0G7WV/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/spdep_1.3-11.tar.gz'
Content type 'application/x-gzip' length 4685359 bytes (4.5 MB)
downloaded 4.5 MB


The downloaded source packages are in
	‘/tmp/Rtmpl0G7WV/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/sf_1.0-20.tar.gz'
Content type 'application/x-gzip' length 4492197 bytes (4.3 MB)
downloaded 4.3 MB


The downloaded 

In [ ]:
%%R
library(INLA)
library(spdep)
library(sf)
library(tidyverse)
library(readxl)


In [ ]:
%%R
# อ่านข้อมูล
data <- read_excel("/content/gdrive/MyDrive/Senior Project/Spatial/DATA/Klebsiella pneumoniae/Levofloxacin/PERCENT_R_Klebsiella pneumoniae_Levofloxacin.xlsx")
data

# A tibble: 1,301 × 6
   GROUP_NAME            ANTIBIOTIC   REGION SPEC_DATE  Year R_percent
   <chr>                 <chr>         <dbl> <chr>     <dbl>     <dbl>
 1 Klebsiella pneumoniae Levofloxacin      1 2015-01    2015      8.33
 2 Klebsiella pneumoniae Levofloxacin      2 2015-01    2015     33.3 
 3 Klebsiella pneumoniae Levofloxacin      5 2015-01    2015     27.2 
 4 Klebsiella pneumoniae Levofloxacin      6 2015-01    2015     22.9 
 5 Klebsiella pneumoniae Levofloxacin      7 2015-01    2015     22.4 
 6 Klebsiella pneumoniae Levofloxacin      8 2015-01    2015     19.0 
 7 Klebsiella pneumoniae Levofloxacin     11 2015-01    2015     26.4 
 8 Klebsiella pneumoniae Levofloxacin     13 2015-01    2015     66.7 
 9 Klebsiella pneumoniae Levofloxacin      1 2015-02    2015     31.8 
10 Klebsiella pneumoniae Levofloxacin      2 2015-02    2015     20.8 
# ℹ 1,291 more rows
# ℹ Use `print(n = ...)` to see more rows


In [ ]:
%%R
# เข้ารหัสพื้นที่ (province) และเวลา (month)
data$Region_id <- as.numeric(as.factor(data$REGION))
# สร้าง factor แล้วแปลงเป็นเลขที่เรียง
data$month_id <- as.numeric(as.factor(data$SPEC_DATE))

data

# A tibble: 1,301 × 8
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <dbl>    <dbl>
 1 Klebsiella pn… Levofloxa…      1 2015-01    2015      8.33         1        1
 2 Klebsiella pn… Levofloxa…      2 2015-01    2015     33.3          2        1
 3 Klebsiella pn… Levofloxa…      5 2015-01    2015     27.2          5        1
 4 Klebsiella pn… Levofloxa…      6 2015-01    2015     22.9          6        1
 5 Klebsiella pn… Levofloxa…      7 2015-01    2015     22.4          7        1
 6 Klebsiella pn… Levofloxa…      8 2015-01    2015     19.0          8        1
 7 Klebsiella pn… Levofloxa…     11 2015-01    2015     26.4         11        1
 8 Klebsiella pn… Levofloxa…     13 2015-01    2015     66.7         13        1
 9 Klebsiella pn… Levofloxa…      1 2015-02    2015     31.8          1        2
10 Klebsiella pn… Levofloxa…      2 2015-02    2015     20.8          2        2
# ℹ 1,

In [ ]:
%%R
unique(data$month_id)

  [1]   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
 [19]  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
 [37]  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
 [55]  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
 [73]  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
 [91]  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108


In [ ]:
%%R
library(sf)
library(spdep)

# โหลด shapefile ของจังหวัด
shp <- st_read("/content/gdrive/MyDrive/Senior Project/Spatial/Regions_no_province_boundaries.json")  # สมมุติว่าคุณมี shapefile แล้ว

# สร้าง adjacency matrix
nb <- poly2nb(shp)
nb2INLA("adjacency.graph", nb)

Reading layer `Regions_no_province_boundaries' from data source 
  `/content/gdrive/.shortcut-targets-by-id/1h5EYQx4w-VInRE_mIX08fjO5fl1RpQIz/Senior Project/Spatial/Regions_no_province_boundaries.json' 
  using driver `GeoJSON'
Simple feature collection with 13 features and 10 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: 97.34336 ymin: 5.613038 xmax: 105.637 ymax: 20.46503
Geodetic CRS:  WGS 84


In [ ]:
%%R
shp

Simple feature collection with 13 features and 10 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: 97.34336 ymin: 5.613038 xmax: 105.637 ymax: 20.46503
Geodetic CRS:  WGS 84
First 10 features:
   HealthRegion   id           ADM1_EN    ADM1_TH  ADM0_EN   ADM0_TH ADM0_PCODE
1             1 <NA>        Chiang Mai    เชียงใหม่ Thailand ประเทศไทย         TH
2             2 <NA>         Uttaradit     อุตรดิตถ์ Thailand ประเทศไทย         TH
3             3 <NA>          Chai Nat      ชัยนาท Thailand ประเทศไทย         TH
4             4 <NA>        Nonthaburi      นนทบุรี Thailand ประเทศไทย         TH
5             5 <NA>        Ratchaburi      ราชบุรี Thailand ประเทศไทย         TH
6             6 <NA>      Samut Prakan สมุทรปราการ Thailand ประเทศไทย         TH
7             7 <NA>         Khon Kaen     ขอนแก่น Thailand ประเทศไทย         TH
8             8 <NA>         Bueng Kan      บึงกาฬ Thailand ประเทศไทย         TH
9             9 <NA> Nakhon Ratchasima  นครราชสีม

In [ ]:
%%R
# แมปข้อมูล %R ของแต่ละจังหวัดใส่ shapefile
shp$HealthRegion_id	 <- as.numeric(as.factor(shp$HealthRegion))
data$Region_id <- match(data$REGION, shp$HealthRegion)

In [ ]:
%%R
data

# A tibble: 1,301 × 8
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Levofloxa…      1 2015-01    2015      8.33         1        1
 2 Klebsiella pn… Levofloxa…      2 2015-01    2015     33.3          2        1
 3 Klebsiella pn… Levofloxa…      5 2015-01    2015     27.2          5        1
 4 Klebsiella pn… Levofloxa…      6 2015-01    2015     22.9          6        1
 5 Klebsiella pn… Levofloxa…      7 2015-01    2015     22.4          7        1
 6 Klebsiella pn… Levofloxa…      8 2015-01    2015     19.0          8        1
 7 Klebsiella pn… Levofloxa…     11 2015-01    2015     26.4         11        1
 8 Klebsiella pn… Levofloxa…     13 2015-01    2015     66.7         13        1
 9 Klebsiella pn… Levofloxa…      1 2015-02    2015     31.8          1        2
10 Klebsiella pn… Levofloxa…      2 2015-02    2015     20.8          2        2
# ℹ 1,

In [ ]:
%%R
unique(data$Region_id)

 [1]  1  2  5  6  7  8 11 13  4 12  3 10  9


In [ ]:
%%R
library(lubridate)

# ดึงเดือนออกมาจาก SPEC_DATE
data$month_num <- month(as.Date(paste0(data$SPEC_DATE, "-01")))

# สร้างตัวแปร sine/cosine สำหรับ seasonal effect
data$sin_month <- sin(2 * pi * data$month_num / 12)
data$cos_month <- cos(2 * pi * data$month_num / 12)


In [ ]:
%%R
library(lubridate)

# ดึงเดือนจาก SPEC_DATE
data$month <- month(as.Date(paste0(data$SPEC_DATE, "-01")))

# แบ่งฤดูกาลแบบไทย (ตัวอย่าง: 3 ฤดู)
data$season <- case_when(
  data$month %in% c(3, 4, 5) ~ "summer",   # มีนาคม–พฤษภาคม
  data$month %in% c(6, 7, 8, 9) ~ "rainy",  # มิถุนายน–กันยายน
  data$month %in% c(10, 11, 12, 1, 2) ~ "winter"  # ต.ค.–ก.พ.
)

# แปลงเป็นรหัสตัวเลข (ID)
data$season_id <- as.numeric(as.factor(data$season))


In [ ]:
%%R
data$region_year <- interaction(data$Region_id, data$Year)

In [ ]:
%%R
data$R_scaled <- (data$R_percent + 0.001) / 100

In [ ]:
%%R
data$R_scaled <- pmin(pmax(data$R_scaled, 0.0001), 0.9999)  # ตัดค่าขอบ

In [ ]:
%%R
data$R_logit <- log(data$R_scaled / (1 - data$R_scaled))  # หรือ: logit(x)

In [ ]:
%%R
# แบ่งชุดข้อมูล
data$set <- ifelse(data$Year %in% 2022:2023, "test", "train")
data

# A tibble: 1,301 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Levofloxa…      1 2015-01    2015      8.33         1        1
 2 Klebsiella pn… Levofloxa…      2 2015-01    2015     33.3          2        1
 3 Klebsiella pn… Levofloxa…      5 2015-01    2015     27.2          5        1
 4 Klebsiella pn… Levofloxa…      6 2015-01    2015     22.9          6        1
 5 Klebsiella pn… Levofloxa…      7 2015-01    2015     22.4          7        1
 6 Klebsiella pn… Levofloxa…      8 2015-01    2015     19.0          8        1
 7 Klebsiella pn… Levofloxa…     11 2015-01    2015     26.4         11        1
 8 Klebsiella pn… Levofloxa…     13 2015-01    2015     66.7         13        1
 9 Klebsiella pn… Levofloxa…      1 2015-02    2015     31.8          1        2
10 Klebsiella pn… Levofloxa…      2 2015-02    2015     20.8          2        2
# ℹ 1

In [ ]:
%%R
train_data <- data[data$set == "train", ]
test_data <- data[data$set == "test", ]
train_data

# A tibble: 995 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Levofloxa…      1 2015-01    2015      8.33         1        1
 2 Klebsiella pn… Levofloxa…      2 2015-01    2015     33.3          2        1
 3 Klebsiella pn… Levofloxa…      5 2015-01    2015     27.2          5        1
 4 Klebsiella pn… Levofloxa…      6 2015-01    2015     22.9          6        1
 5 Klebsiella pn… Levofloxa…      7 2015-01    2015     22.4          7        1
 6 Klebsiella pn… Levofloxa…      8 2015-01    2015     19.0          8        1
 7 Klebsiella pn… Levofloxa…     11 2015-01    2015     26.4         11        1
 8 Klebsiella pn… Levofloxa…     13 2015-01    2015     66.7         13        1
 9 Klebsiella pn… Levofloxa…      1 2015-02    2015     31.8          1        2
10 Klebsiella pn… Levofloxa…      2 2015-02    2015     20.8          2        2
# ℹ 985

In [ ]:
%%R
test_data$R_logit_real <- test_data$R_logit# เก็บของจริงไว้ใช้เทียบภายหลัง
test_data$R_logit <- NA  # ให้โมเดลพยากรณ์
test_data$R_logit_real

  [1] -0.44420913 -9.21024037 -0.71874422 -0.79846097 -0.50891207 -0.45194305
  [7] -0.18054361 -0.91624173  0.09101186 -0.11118551 -0.91127082 -0.40542344
 [13] -0.60369684 -0.94345371 -9.21024037 -0.96243011 -1.29922359 -0.73658662
 [19] -1.79167780 -0.41647318 -0.60020972  0.03394156  0.10802875 -1.44685413
 [25] -0.33643109 -0.74189158 -1.07904666 -0.84725024 -1.11961180 -1.35448433
 [31] -1.08329189 -0.31011396 -0.33447224 -0.48900938 -0.26506704 -0.14864298
 [37] -1.18481687 -0.84151968 -1.02835649 -0.92288613 -9.21024037 -0.71977002
 [43] -0.88234088 -0.76800229 -2.56479860 -0.08434732 -0.60324710 -0.28219167
 [49] -0.16562685 -1.03296335 -0.46698078 -0.98936238 -1.44685413 -1.60936591
 [55] -1.17345817 -0.87078028 -0.69799209 -1.67390123 -0.31909600 -0.82985883
 [61]  0.01530747 -0.05855413 -1.03360272 -0.29294626 -0.91284621 -1.64013607
 [67] -9.21024037 -0.75681699 -1.98090759 -1.02553568 -1.38623186 -0.21821309
 [73] -0.61175768 -0.27439609 -0.03302085 -0.80906015 -1.0018317

In [ ]:
%%R
# Add the R_scaled_real column to train_data and fill it with NAs
train_data$R_logit_real <- NA

# Now, combine the data frames
combined_data <- rbind(train_data, test_data)
combined_data

# A tibble: 1,301 × 19
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Levofloxa…      1 2015-01    2015      8.33         1        1
 2 Klebsiella pn… Levofloxa…      2 2015-01    2015     33.3          2        1
 3 Klebsiella pn… Levofloxa…      5 2015-01    2015     27.2          5        1
 4 Klebsiella pn… Levofloxa…      6 2015-01    2015     22.9          6        1
 5 Klebsiella pn… Levofloxa…      7 2015-01    2015     22.4          7        1
 6 Klebsiella pn… Levofloxa…      8 2015-01    2015     19.0          8        1
 7 Klebsiella pn… Levofloxa…     11 2015-01    2015     26.4         11        1
 8 Klebsiella pn… Levofloxa…     13 2015-01    2015     66.7         13        1
 9 Klebsiella pn… Levofloxa…      1 2015-02    2015     31.8          1        2
10 Klebsiella pn… Levofloxa…      2 2015-02    2015     20.8          2        2
# ℹ 1

In [ ]:
%%R

formula <- R_logit ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")+ sin_month + cos_month + f(region_year, model="iid")
#formula <- R_logit ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1") +  f(region_year, model="iid")
result <- inla(formula,
               family="gaussian",
               data=combined_data,
               control.predictor=list(compute=TRUE),
               control.compute=list(dic=TRUE),
               control.inla = list(strategy = "simplified.laplace"))

# ดูผลลัพธ์
summary(result)

Time used:
    Pre = 15.7, Running = 2.56, Post = 0.068, Total = 18.4 
Fixed effects:
              mean    sd 0.025quant 0.5quant 0.975quant   mode kld
(Intercept) -0.915 0.183     -1.276   -0.915     -0.554 -0.915   0
sin_month   -0.053 0.051     -0.154   -0.053      0.047 -0.053   0
cos_month    0.000 0.048     -0.094    0.000      0.094  0.000   0

Random effects:
  Name	  Model
    Region_id BYM2 model
   month_id RW2 model
   Year RW1 model
   region_year IID model

Model hyperparameters:
                                            mean       sd 0.025quant 0.5quant
Precision for the Gaussian observations 8.95e-01 4.60e-02      0.809 8.94e-01
Precision for Region_id                 8.21e+00 6.53e+00      1.792 6.40e+00
Phi for Region_id                       3.04e-01 2.28e-01      0.020 2.48e-01
Precision for month_id                  3.45e+04 2.71e+04   5387.771 2.72e+04
Precision for Year                      2.02e+04 1.82e+04   1263.712 1.46e+04
Precision for region_year       

In [ ]:
%%R
test_data$predicted_logit <- result$summary.fitted.values$mean[
  (nrow(train_data) + 1):nrow(data)
]

In [ ]:
%%R
test_data$predicted_scaled <- plogis(test_data$predicted_logit)  # inverse logit


In [ ]:
%%R
test_data$predicted_percent <- test_data$predicted_scaled * 100
test_data$actual_percent    <- test_data$R_percent

In [ ]:
%%R
test_data$actual_percent

  [1]  39.072848   0.000000  32.765957  31.034483  37.543860  38.888889
  [7]  45.497630  28.571429  52.272727  47.222222  28.672986  40.000000
 [13]  35.348837  28.019324   0.000000  27.638191  21.428571  32.374101
 [19]  14.285714  39.735099  35.428571  50.847458  52.697095  19.047619
 [25]  41.666667  32.258065  25.367647  30.000000  24.607330  20.512821
 [31]  25.287356  42.307692  41.714286  38.011696  43.410853  46.289753
 [37]  23.417722  30.120482  26.339286  28.436019   0.000000  32.743363
 [43]  29.268293  31.690141   7.142857  47.891566  35.359116  42.990654
 [49]  45.867769  26.250000  38.532110  27.102804  19.047619  16.666667
 [55]  23.622047  29.508197  33.224756  15.789474  42.088608  30.366492
 [61]  50.381679  48.535565  26.237624  42.727273  28.640777  16.243655
 [67]   0.000000  31.932773  12.121212  26.394052  20.000000  44.565217
 [73]  35.164835  43.181818  49.173554  30.808081  26.857143  31.100478
 [79]  28.959276  36.363636  25.217391  14.925373  15.384615  49

In [ ]:
%%R
test_data$predicted_percent

  [1] 15.34511 24.35609 23.98140 37.46965 24.60829 24.75298 22.55872 25.42805
  [9] 18.63717 26.72167 24.87478 33.28048 32.85517 15.02973 23.90784 23.53790
 [17] 36.89779 24.15688 24.29977 22.13386 24.96659 18.26874 26.24499 24.42006
 [25] 32.73906 32.31734 14.87633 23.68912 23.32151 36.61743 23.93659 24.07859
 [33] 21.92669 24.74133 18.08933 26.01220 24.19813 32.47402 32.05411 14.90465
 [41] 23.72956 23.36152 36.66936 23.97732 24.11948 21.96498 24.78299 18.12246
 [49] 26.05526 24.23916 32.52306 32.10285 15.09045 23.99438 23.62349 37.00854
 [57] 24.24402 24.38726 22.21584 25.05570 18.33974 26.33708 24.50783 32.84376
 [65] 32.42141 15.36998 24.39149 24.01640 37.51476 24.64392 24.78875 22.59226
 [73] 25.46450 18.66622 26.75932 24.91065 33.32311 32.89765 15.65265 24.79149
 [81] 24.41223 25.04668 25.19309 22.97171 25.87608 18.99590 27.18420 25.31631
 [89] 33.80410 33.37560 15.84303 25.06004 24.67801 25.31706 25.46451 23.22663
 [97] 26.15229 19.21768 27.46921 25.58860 34.12598 33.69550 15.8

In [ ]:
%%R
wape <- sum(abs(test_data$predicted_percent - test_data$actual_percent)) /
        sum(abs(test_data$actual_percent))

wape_percent <- round(wape * 100, 2)

cat("WAPE for 2022–2023 =", wape_percent, "%\n")


WAPE for 2022–2023 = 32.25 %


In [ ]:
%%R

test_data_valid <- test_data[test_data$actual_percent > 0, ]

# คำนวณ MAPE
mape_val <- mean(abs(test_data_valid$predicted_percent - test_data_valid$actual_percent) /
                 test_data_valid$actual_percent) * 100

# แสดงผล
cat("MAPE =", round(mape_val, 2), "%\n")


MAPE = 29.9 %


##Predict

In [ ]:
%%R
data

# A tibble: 1,391 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Acinetobacter… Gentamicin      1 2015-01    2015      62.2         1        1
 2 Acinetobacter… Gentamicin      2 2015-01    2015      81.0         2        1
 3 Acinetobacter… Gentamicin      3 2015-01    2015      53.8         3        1
 4 Acinetobacter… Gentamicin      4 2015-01    2015      76.2         4        1
 5 Acinetobacter… Gentamicin      5 2015-01    2015      70.8         5        1
 6 Acinetobacter… Gentamicin      6 2015-01    2015      69.2         6        1
 7 Acinetobacter… Gentamicin      7 2015-01    2015      72.9         7        1
 8 Acinetobacter… Gentamicin      8 2015-01    2015      58.4         8        1
 9 Acinetobacter… Gentamicin      9 2015-01    2015      85.8         9        1
10 Acinetobacter… Gentamicin     10 2015-01    2015      64.3        10        1
# ℹ 1

In [ ]:
%%R
data

# A tibble: 1,391 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Acinetobacter… Gentamicin      1 2015-01    2015      62.2         1        1
 2 Acinetobacter… Gentamicin      2 2015-01    2015      81.0         2        1
 3 Acinetobacter… Gentamicin      3 2015-01    2015      53.8         3        1
 4 Acinetobacter… Gentamicin      4 2015-01    2015      76.2         4        1
 5 Acinetobacter… Gentamicin      5 2015-01    2015      70.8         5        1
 6 Acinetobacter… Gentamicin      6 2015-01    2015      69.2         6        1
 7 Acinetobacter… Gentamicin      7 2015-01    2015      72.9         7        1
 8 Acinetobacter… Gentamicin      8 2015-01    2015      58.4         8        1
 9 Acinetobacter… Gentamicin      9 2015-01    2015      85.8         9        1
10 Acinetobacter… Gentamicin     10 2015-01    2015      64.3        10        1
# ℹ 1

In [ ]:
%%R
future_years <- 2024:2028
future_months <- 1:12
future_regions <- sort(unique(data$REGION))  # หรือ Health Region ที่มี
future_GROUP_NAME <- unique(data$GROUP_NAME)
future_ANTIBIOTIC <- unique(data$ANTIBIOTIC)
# สร้าง grid ของข้อมูลล่วงหน้า
future_data <- expand.grid(
  GROUP_NAME = future_GROUP_NAME,
  ANTIBIOTIC = future_ANTIBIOTIC,
  REGION = future_regions,
  month = future_months,
  Year = future_years
)
future_data

                 GROUP_NAME ANTIBIOTIC REGION month Year
1   Acinetobacter baumannii Gentamicin      1     1 2024
2   Acinetobacter baumannii Gentamicin      2     1 2024
3   Acinetobacter baumannii Gentamicin      3     1 2024
4   Acinetobacter baumannii Gentamicin      4     1 2024
5   Acinetobacter baumannii Gentamicin      5     1 2024
6   Acinetobacter baumannii Gentamicin      6     1 2024
7   Acinetobacter baumannii Gentamicin      7     1 2024
8   Acinetobacter baumannii Gentamicin      8     1 2024
9   Acinetobacter baumannii Gentamicin      9     1 2024
10  Acinetobacter baumannii Gentamicin     10     1 2024
11  Acinetobacter baumannii Gentamicin     11     1 2024
12  Acinetobacter baumannii Gentamicin     12     1 2024
13  Acinetobacter baumannii Gentamicin     13     1 2024
14  Acinetobacter baumannii Gentamicin      1     2 2024
15  Acinetobacter baumannii Gentamicin      2     2 2024
16  Acinetobacter baumannii Gentamicin      3     2 2024
17  Acinetobacter baumannii Gen

In [ ]:
%%R
library(dplyr)

future_data <- future_data %>%
  mutate(SPEC_DATE = sprintf("%d-%02d", Year, month))
future_data

                 GROUP_NAME ANTIBIOTIC REGION month Year SPEC_DATE
1   Acinetobacter baumannii Gentamicin      1     1 2024   2024-01
2   Acinetobacter baumannii Gentamicin      2     1 2024   2024-01
3   Acinetobacter baumannii Gentamicin      3     1 2024   2024-01
4   Acinetobacter baumannii Gentamicin      4     1 2024   2024-01
5   Acinetobacter baumannii Gentamicin      5     1 2024   2024-01
6   Acinetobacter baumannii Gentamicin      6     1 2024   2024-01
7   Acinetobacter baumannii Gentamicin      7     1 2024   2024-01
8   Acinetobacter baumannii Gentamicin      8     1 2024   2024-01
9   Acinetobacter baumannii Gentamicin      9     1 2024   2024-01
10  Acinetobacter baumannii Gentamicin     10     1 2024   2024-01
11  Acinetobacter baumannii Gentamicin     11     1 2024   2024-01
12  Acinetobacter baumannii Gentamicin     12     1 2024   2024-01
13  Acinetobacter baumannii Gentamicin     13     1 2024   2024-01
14  Acinetobacter baumannii Gentamicin      1     2 2024   202

In [ ]:
%%R
# เข้ารหัสพื้นที่ (province) และเวลา (month)
future_data$Region_id <- as.numeric(as.factor(future_data$REGION))
# สร้าง factor แล้วแปลงเป็นเลขที่เรียง
future_data$month_id <- as.numeric(as.factor(future_data$SPEC_DATE))

# สร้าง region-year interaction (ถ้าใช้ในโมเดล)
future_data$region_year <- interaction(future_data$Region_id, future_data$Year)

# เติมค่าที่ยังไม่มี (target)
future_data$R_logit <- NA  # หรือ R_scaled หรือ R_percent ตามที่ใช้
future_data$R_scaled <- NA
future_data$R_percent<- NA
head(future_data,10)

                GROUP_NAME ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Acinetobacter baumannii Gentamicin      1     1 2024   2024-01         1
2  Acinetobacter baumannii Gentamicin      2     1 2024   2024-01         2
3  Acinetobacter baumannii Gentamicin      3     1 2024   2024-01         3
4  Acinetobacter baumannii Gentamicin      4     1 2024   2024-01         4
5  Acinetobacter baumannii Gentamicin      5     1 2024   2024-01         5
6  Acinetobacter baumannii Gentamicin      6     1 2024   2024-01         6
7  Acinetobacter baumannii Gentamicin      7     1 2024   2024-01         7
8  Acinetobacter baumannii Gentamicin      8     1 2024   2024-01         8
9  Acinetobacter baumannii Gentamicin      9     1 2024   2024-01         9
10 Acinetobacter baumannii Gentamicin     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent
1         1      1.2024      NA       NA        NA
2         1      2.2024      NA       NA        NA
3         1

In [ ]:
%%R
library(lubridate)

# ดึงเดือนออกมาจาก SPEC_DATE
future_data$month_num <- month(as.Date(paste0(future_data$SPEC_DATE, "-01")))

# สร้างตัวแปร sine/cosine สำหรับ seasonal effect
future_data$sin_month <- sin(2 * pi * future_data$month_num / 12)
future_data$cos_month <- cos(2 * pi * future_data$month_num / 12)
head(future_data,10)

                GROUP_NAME ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Acinetobacter baumannii Gentamicin      1     1 2024   2024-01         1
2  Acinetobacter baumannii Gentamicin      2     1 2024   2024-01         2
3  Acinetobacter baumannii Gentamicin      3     1 2024   2024-01         3
4  Acinetobacter baumannii Gentamicin      4     1 2024   2024-01         4
5  Acinetobacter baumannii Gentamicin      5     1 2024   2024-01         5
6  Acinetobacter baumannii Gentamicin      6     1 2024   2024-01         6
7  Acinetobacter baumannii Gentamicin      7     1 2024   2024-01         7
8  Acinetobacter baumannii Gentamicin      8     1 2024   2024-01         8
9  Acinetobacter baumannii Gentamicin      9     1 2024   2024-01         9
10 Acinetobacter baumannii Gentamicin     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent month_num sin_month
1         1      1.2024      NA       NA        NA         1       0.5
2         1      2.202

In [ ]:
%%R
library(lubridate)

# ดึงเดือนจาก SPEC_DATE
future_data$month <- month(as.Date(paste0(future_data$SPEC_DATE, "-01")))

# แบ่งฤดูกาลแบบไทย (ตัวอย่าง: 3 ฤดู)
future_data$season <- case_when(
  future_data$month %in% c(3, 4, 5) ~ "summer",   # มีนาคม–พฤษภาคม
  future_data$month %in% c(6, 7, 8, 9) ~ "rainy",  # มิถุนายน–กันยายน
  future_data$month %in% c(10, 11, 12, 1, 2) ~ "winter"  # ต.ค.–ก.พ.
)

# แปลงเป็นรหัสตัวเลข (ID)
future_data$season_id <- as.numeric(as.factor(future_data$season))
head(future_data,10)

                GROUP_NAME ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Acinetobacter baumannii Gentamicin      1     1 2024   2024-01         1
2  Acinetobacter baumannii Gentamicin      2     1 2024   2024-01         2
3  Acinetobacter baumannii Gentamicin      3     1 2024   2024-01         3
4  Acinetobacter baumannii Gentamicin      4     1 2024   2024-01         4
5  Acinetobacter baumannii Gentamicin      5     1 2024   2024-01         5
6  Acinetobacter baumannii Gentamicin      6     1 2024   2024-01         6
7  Acinetobacter baumannii Gentamicin      7     1 2024   2024-01         7
8  Acinetobacter baumannii Gentamicin      8     1 2024   2024-01         8
9  Acinetobacter baumannii Gentamicin      9     1 2024   2024-01         9
10 Acinetobacter baumannii Gentamicin     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent month_num sin_month
1         1      1.2024      NA       NA        NA         1       0.5
2         1      2.202

In [ ]:
%%R
future_data$set <- NA
head(future_data,10)

                GROUP_NAME ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Acinetobacter baumannii Gentamicin      1     1 2024   2024-01         1
2  Acinetobacter baumannii Gentamicin      2     1 2024   2024-01         2
3  Acinetobacter baumannii Gentamicin      3     1 2024   2024-01         3
4  Acinetobacter baumannii Gentamicin      4     1 2024   2024-01         4
5  Acinetobacter baumannii Gentamicin      5     1 2024   2024-01         5
6  Acinetobacter baumannii Gentamicin      6     1 2024   2024-01         6
7  Acinetobacter baumannii Gentamicin      7     1 2024   2024-01         7
8  Acinetobacter baumannii Gentamicin      8     1 2024   2024-01         8
9  Acinetobacter baumannii Gentamicin      9     1 2024   2024-01         9
10 Acinetobacter baumannii Gentamicin     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent month_num sin_month
1         1      1.2024      NA       NA        NA         1       0.5
2         1      2.202

In [ ]:
%%R
head(data,10)

# A tibble: 10 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Acinetobacter… Gentamicin      1 2015-01    2015      62.2         1        1
 2 Acinetobacter… Gentamicin      2 2015-01    2015      81.0         2        1
 3 Acinetobacter… Gentamicin      3 2015-01    2015      53.8         3        1
 4 Acinetobacter… Gentamicin      4 2015-01    2015      76.2         4        1
 5 Acinetobacter… Gentamicin      5 2015-01    2015      70.8         5        1
 6 Acinetobacter… Gentamicin      6 2015-01    2015      69.2         6        1
 7 Acinetobacter… Gentamicin      7 2015-01    2015      72.9         7        1
 8 Acinetobacter… Gentamicin      8 2015-01    2015      58.4         8        1
 9 Acinetobacter… Gentamicin      9 2015-01    2015      85.8         9        1
10 Acinetobacter… Gentamicin     10 2015-01    2015      64.3        10        1
# ℹ 10 m

In [ ]:
%%R
combined_data_future <- rbind(data, future_data)
combined_data_future

# A tibble: 2,171 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <dbl>    <dbl>
 1 Acinetobacter… Gentamicin      1 2015-01    2015      62.2         1        1
 2 Acinetobacter… Gentamicin      2 2015-01    2015      81.0         2        1
 3 Acinetobacter… Gentamicin      3 2015-01    2015      53.8         3        1
 4 Acinetobacter… Gentamicin      4 2015-01    2015      76.2         4        1
 5 Acinetobacter… Gentamicin      5 2015-01    2015      70.8         5        1
 6 Acinetobacter… Gentamicin      6 2015-01    2015      69.2         6        1
 7 Acinetobacter… Gentamicin      7 2015-01    2015      72.9         7        1
 8 Acinetobacter… Gentamicin      8 2015-01    2015      58.4         8        1
 9 Acinetobacter… Gentamicin      9 2015-01    2015      85.8         9        1
10 Acinetobacter… Gentamicin     10 2015-01    2015      64.3        10        1
# ℹ 2

In [ ]:
%%R
# จากนั้นใช้ formula แบบ besag ได้เลย
formula <- R_logit ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")+ sin_month + cos_month + f(region_year, model="iid")
# formula <- R_logit ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")
result <- inla(formula,
               family="gaussian",
               data=combined_data_future,
               control.predictor=list(compute=TRUE),
               control.compute=list(dic=TRUE),
               control.inla = list(strategy = "simplified.laplace"))

# ดูผลลัพธ์
summary(result)

Time used:
    Pre = 27.2, Running = 3.83, Post = 0.123, Total = 31.1 
Fixed effects:
             mean    sd 0.025quant 0.5quant 0.975quant  mode kld
(Intercept) 0.758 0.159      0.482    0.746      1.101 0.722   0
sin_month   0.096 0.023      0.049    0.096      0.140 0.096   0
cos_month   0.138 0.016      0.108    0.138      0.169 0.138   0

Random effects:
  Name	  Model
    Region_id BYM2 model
   month_id RW2 model
   Year RW1 model
   region_year IID model

Model hyperparameters:
                                            mean       sd 0.025quant 0.5quant
Precision for the Gaussian observations 6.81e+00 2.73e-01      6.295 6.81e+00
Precision for Region_id                 2.51e+01 1.20e+01      9.237 2.27e+01
Phi for Region_id                       4.84e-01 2.47e-01      0.074 4.75e-01
Precision for month_id                  3.91e+04 2.66e+04   8826.734 3.25e+04
Precision for Year                      3.78e+01 3.29e+01      7.522 2.85e+01
Precision for region_year               

In [ ]:
%%R
# คำนวณจำนวนแถว
n_total       <- nrow(combined_data_future)
n_future      <- nrow(future_data)
start_index   <- n_total - n_future + 1
end_index     <- n_total

# ดึงค่าที่พยากรณ์เฉพาะ future
future_data$predicted_logit <- result$summary.fitted.values$mean[start_index:end_index]

In [ ]:
%%R
# inverse logit
future_data$predicted_scaled  <- plogis(future_data$predicted_logit)

# scale เป็น %
future_data$predicted_percent <- future_data$predicted_scaled * 100

In [ ]:
%%R
head(future_data[, c("REGION", "SPEC_DATE", "predicted_percent")],13)

   REGION SPEC_DATE predicted_percent
1       1   2024-01          83.25176
2       2   2024-01          86.97776
3       3   2024-01          85.81682
4       4   2024-01          87.43636
5       5   2024-01          87.33009
6       6   2024-01          87.63891
7       7   2024-01          87.85278
8       8   2024-01          85.77100
9       9   2024-01          87.77766
10     10   2024-01          81.73655
11     11   2024-01          80.42447
12     12   2024-01          78.61712
13     13   2024-01          84.22896


In [ ]:
%%R
install.packages("writexl")  # รันครั้งเดียว
library(writexl)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/writexl_1.5.3.tar.gz'
Content type 'application/x-gzip' length 278300 bytes (271 KB)
downloaded 271 KB


The downloaded source packages are in
	‘/tmp/RtmpMeFqEZ/downloaded_packages’


In [ ]:
%%R
write_xlsx(future_data, path = "/content/gdrive/MyDrive/Senior Project/Spatial/Spatiotemporal/results/Klebsiella pneumoniae/Levofloxacin/Future_R_Klebsiella pneumoniae_Levofloxacin.xlsx")